# 09 â€” Full Evaluation (LOCAL)
## WavSent-MTL Â· Tasks 4.1â€“4.16

**Runs on: Local PC**

**What this notebook does:**
- Merge all ablation results (configs Aâ€“G) for both datasets
- Aggregate: mean, max, std per config across 30 seeds
- Statistical tests: Wilcoxon (A vs G), Shapiro-Wilk normality check
- Plots: ablation comparison, confusion matrix, AUC-ROC, loss curves
- SHAP analysis on best config
- Granger causality: polarity_mean vs returns (lags 1â€“5)
- Trading simulation (long-only) + Sharpe ratios
- Baselines: SVM + RF
- Save all figures â†’ results/figures/ and tables â†’ results/tables/

**PREREQUISITE:**
- Download from Kaggle: kotekar_ablation_AG.csv, kaggle_ablation_AG.csv
- Download: pso_weights_*.json, ensemble_results_*.csv
- Place all in ablation/results/kotekar/ and ablation/results/kaggle/
- Download .npy arrays (val_predictions/) from Kaggle outputs

In [1]:
import sys
import os

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import json
import matplotlib
matplotlib.use('Agg')   # non-interactive backend
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from config.config import CONFIG

# Output directories
for dataset in ['kotekar', 'kaggle']:
    os.makedirs(os.path.join(project_root, CONFIG['figures_dir'], dataset), exist_ok=True)
    os.makedirs(os.path.join(project_root, CONFIG['tables_dir'], dataset), exist_ok=True)

print('CONFIG loaded.')
print('Benchmark: Kotekar et al. accuracy=0.5853, Sharpe=1.5679')

CONFIG loaded.
Benchmark: Kotekar et al. accuracy=0.5853, Sharpe=1.5679


## Task 4.1 â€” Load and Merge All Ablation Results

In [2]:
ABLATION_DIR = os.path.join(project_root, CONFIG['ablation_dir'])

results = {}
for dataset in ['kotekar', 'kaggle']:
    # Main ablation CSV (configs A-F)
    main_csv = os.path.join(ABLATION_DIR, dataset, f'{dataset}_ablation_AG.csv')
    df_main = pd.read_csv(main_csv)

    # Config G (ensemble)
    g_csv = os.path.join(ABLATION_DIR, dataset, f'ensemble_results_{dataset}.csv')
    # Also check working directory from Kaggle download
    if not os.path.exists(g_csv):
        g_csv = os.path.join(project_root, f'ensemble_results_{dataset}.csv')
    df_g = pd.read_csv(g_csv) if os.path.exists(g_csv) else pd.DataFrame()

    df_all = pd.concat([df_main, df_g], ignore_index=True)
    results[dataset] = df_all

    print(f'{dataset}: {len(df_all)} rows | configs: {sorted(df_all["config"].unique())}')

results['kotekar'].head(3)

kotekar: 181 rows | configs: ['A', 'B', 'C', 'D', 'E', 'F', 'G']
kaggle: 181 rows | configs: ['A', 'B', 'C', 'D', 'E', 'F', 'G']


,config,model,seed,run,dataset,accuracy,balanced_accuracy,auc,precision,recall,f1,rmse,mae,r2,val_accuracy
0,A,tkan,0,0,kotekar,0.564103,0.5,0.489806,0.564103,1.0,0.721311,0.406009,0.331836,-0.012508,0.6
1,A,tkan,1,1,kotekar,0.564103,0.5,0.520053,0.564103,1.0,0.721311,0.403611,0.324890,-0.000580,0.6
2,A,tkan,2,2,kotekar,0.564103,0.5,0.437667,0.564103,1.0,0.721311,0.407222,0.333911,-0.018566,0.6


## Task 4.2 â€” Aggregate: Mean, Max, Std per Config

In [3]:
from src.evaluation.metrics import aggregate_run_metrics

METRIC_COLS = ['accuracy', 'balanced_accuracy', 'auc', 'precision', 'recall', 'f1', 'rmse', 'mae', 'r2']

for dataset in ['kotekar', 'kaggle']:
    df = results[dataset]
    rows = []
    for cfg in sorted(df['config'].unique()):
        cfg_df = df[df['config'] == cfg]
        for metric in METRIC_COLS:
            if metric in cfg_df.columns:
                rows.append({
                    'config': cfg,
                    'metric': metric,
                    'mean':   cfg_df[metric].mean(),
                    'max':    cfg_df[metric].max(),
                    'std':    cfg_df[metric].std(),
                })

    summary_df = pd.DataFrame(rows)
    out_path = os.path.join(project_root, CONFIG['tables_dir'], dataset, 'ablation_summary.csv')
    summary_df.to_csv(out_path, index=False)
    print(f'{dataset} summary saved: {out_path}')

    # Print accuracy summary
    acc_summary = summary_df[summary_df['metric'] == 'accuracy'][['config','mean','max','std']]
    print(f'{dataset.upper()} ï¿½ Accuracy across configs:')
    print(acc_summary.to_string(index=False))
    print()

    # Print regression metrics summary
    for reg_metric in ['rmse', 'mae', 'r2']:
        reg_df = summary_df[summary_df['metric'] == reg_metric]
        if not reg_df.empty:
            print(f'{dataset.upper()} ï¿½ {reg_metric.upper()} across configs:')
            print(reg_df[['config','mean','max','std']].to_string(index=False))
            print()

kotekar summary saved: d:/WavSent-MTL/results/tables/kotekar\ablation_summary.csv
KOTEKAR ï¿½ Accuracy across configs:
config     mean      max      std
     A 0.564103 0.564103 0.000000
     B 0.516239 0.564103 0.060480
     C 0.544231 0.589744 0.020295
     D 0.572436 0.628205 0.016247
     E 0.592094 0.647436 0.019739
     F 0.574359 0.641026 0.028199
     G 0.564103 0.564103      NaN

KOTEKAR ï¿½ RMSE across configs:
config     mean      max      std
     A 0.405268 0.410289 0.002158
     B 0.407041 0.418221 0.003564
     C 0.360015 0.368081 0.004358
     D 0.390169 0.421723 0.022885
     E 0.368043 0.387993 0.009104
     F 0.371957 0.395176 0.012915
     G      NaN      NaN      NaN

KOTEKAR ï¿½ MAE across configs:
config     mean      max      std
     A 0.329464 0.338137 0.004398
     B 0.333086 0.346451 0.005804
     C 0.285905 0.294201 0.005604
     D 0.313903 0.349644 0.021192
     E 0.287951 0.300194 0.007017
     F 0.289172 0.305408 0.008877
     G      NaN      NaN      Na

## Task 4.3 â€” Wilcoxon Signed-Rank Test (Config A vs Config G)

In [ ]:
from scipy.stats import wilcoxon, shapiro
import numpy as np

wilcoxon_results = {}

for dataset in ['kotekar', 'kaggle']:
    df = results[dataset]
    acc_A = df[df['config'] == CONFIG['wilcoxon_config_a']]['accuracy'].values
    acc_G = df[df['config'] == CONFIG['wilcoxon_config_b']]['accuracy'].values

    print(f'
--- {dataset.upper()} ---')

    # Task 4.4: Shapiro-Wilk normality check
    if len(acc_A) >= 3:
        stat_s, p_s = shapiro(acc_A)
        print(f'Shapiro-Wilk Config A: stat={stat_s:.4f}, p={p_s:.4f} -> {"normal" if p_s > 0.05 else "non-normal"}')

    if len(acc_A) >= 2 and len(acc_G) >= 1:
        std_A  = float(np.std(acc_A))
        mean_A = float(np.mean(acc_A))
        g_val  = float(acc_G[0])

        if std_A == 0.0:
            # Config A collapsed to a single constant value (all 30 seeds identical).
            # Wilcoxon / t-test both require non-zero variance — use descriptive comparison.
            if abs(mean_A - g_val) < 1e-8:
                note = (f'Config A constant ({mean_A:.4f}) == Config G ({g_val:.4f}); '
                        f'identical performance - not significant')
                stat, p, sig = None, None, False
            else:
                # All differences same sign -> sign test p ~ 0
                note = (f'Config A constant ({mean_A:.4f}), Config G ({g_val:.4f}); '
                        f'zero-variance Config A - sign test trivially significant')
                stat, p, sig = None, 0.0, True
            print(f'(Note: Zero-variance in Config A - descriptive comparison used)')
            print(f'  {note}')
        else:
            from scipy.stats import ttest_1samp
            stat, p_val = ttest_1samp(acc_A, g_val)
            p   = float(p_val)
            sig = p < 0.05
            note = '1-sample t-test (Config G is a single run)'
            print(f'(Note: Using 1-sample t-test since Config G has 1 run)')
            print(f'  stat={stat:.4f}, p={p:.4f}')

        significance = 'SIGNIFICANT' if sig else 'not significant'
        print(f'Config A vs G -> {significance}')

        wilcoxon_results[dataset] = {
            'config_a':   'A',
            'config_b':   'G',
            'mean_A':     round(mean_A, 4),
            'val_G':      round(g_val,  4),
            'stat':       round(stat, 4) if stat is not None else None,
            'p_value':    round(p,    4) if p    is not None else None,
            'significant': sig,
            'note':       note
        }

# Save to wilcoxon_results.csv (separate file - not granger_results.csv)
for dataset, row in wilcoxon_results.items():
    out = os.path.join(project_root, CONFIG['tables_dir'], dataset, 'wilcoxon_results.csv')
    pd.DataFrame([row]).to_csv(out, index=False)
    print(f'Saved: {out}')


## Task 4.5 â€” Ablation Comparison Plot

In [5]:
for dataset in ['kotekar', 'kaggle']:
    df = results[dataset]
    cfg_order = [c for c in ['A','B','C','D','E','F','G'] if c in df['config'].unique()]
    means = [df[df['config']==c]['accuracy'].mean() for c in cfg_order]
    stds  = [df[df['config']==c]['accuracy'].std() for c in cfg_order]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(cfg_order, means, yerr=stds, capsize=4,
                  color=['#90CAF9']*3 + ['#4CAF50']*3 + ['#E53935'],
                  edgecolor='black', linewidth=0.5)
    ax.axhline(0.5853, color='red', linestyle='--', linewidth=1.5,
               label='Kotekar benchmark (0.5853)')
    ax.set_xlabel('Configuration')
    ax.set_ylabel('Test Accuracy')
    ax.set_title(f'Ablation Study â€” {dataset.capitalize()} Dataset')
    ax.legend()
    ax.set_ylim(0.45, max(means) + 0.05)

    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{m:.3f}', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    out_path = os.path.join(project_root, CONFIG['figures_dir'], dataset, 'ablation_comparison.png')
    fig.savefig(out_path, dpi=150)
    plt.close()
    print(f'Saved: {out_path}')

Saved: d:/WavSent-MTL/results/figures/kotekar\ablation_comparison.png
Saved: d:/WavSent-MTL/results/figures/kaggle\ablation_comparison.png


## Task 4.6 â€” Confusion Matrix (Config G, Kotekar)

In [6]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

for dataset in ['kotekar', 'kaggle']:
    # Load ensemble test predictions
    pred_dir = os.path.join(ABLATION_DIR, dataset, 'val_predictions')
    weights_path = os.path.join(ABLATION_DIR, dataset, f'pso_weights_{dataset}.json')
    test_npy = os.path.join(project_root, CONFIG[f'{dataset}_processed_dir'], 'y_clf_test.npy')

    if not os.path.exists(weights_path) or not os.path.exists(test_npy):
        print(f'{dataset}: skipping confusion matrix (files not found)')
        continue

    with open(weights_path) as f:
        weights = json.load(f)

    y_test = np.load(test_npy)

    # Reconstruct ensemble test predictions
    ensemble_probs = np.zeros(len(y_test), dtype=np.float32)
    for model_name in CONFIG['pso_models']:
        test_pred_path = os.path.join(pred_dir, f'{model_name}_test_preds.npy')
        if os.path.exists(test_pred_path):
            ensemble_probs += weights.get(model_name, 0) * np.load(test_pred_path)

    y_pred = (ensemble_probs >= 0.5).astype(int)
    cm = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Down', 'Up'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f'Confusion Matrix â€” Config G â€” {dataset.capitalize()}')
    plt.tight_layout()
    out_path = os.path.join(project_root, CONFIG['figures_dir'], dataset, 'confusion_matrix.png')
    fig.savefig(out_path, dpi=150)
    plt.close()
    print(f'Saved: {out_path}')

Saved: d:/WavSent-MTL/results/figures/kotekar\confusion_matrix.png
Saved: d:/WavSent-MTL/results/figures/kaggle\confusion_matrix.png


## Task 4.7 â€” AUC-ROC Curves

In [7]:
from sklearn.metrics import roc_curve, auc

for dataset in ['kotekar', 'kaggle']:
    pred_dir = os.path.join(ABLATION_DIR, dataset, 'val_predictions')
    weights_path = os.path.join(ABLATION_DIR, dataset, f'pso_weights_{dataset}.json')
    test_npy = os.path.join(project_root, CONFIG[f'{dataset}_processed_dir'], 'y_clf_test.npy')

    if not os.path.exists(weights_path) or not os.path.exists(test_npy):
        print(f'{dataset}: skipping AUC-ROC (files not found)')
        continue

    with open(weights_path) as f:
        weights = json.load(f)
    y_test = np.load(test_npy)

    fig, ax = plt.subplots(figsize=(7, 6))

    # Individual model curves
    colors = {'tkan': '#1976D2', 'lstm': '#388E3C', 'gru': '#F57C00', 'tcn': '#7B1FA2'}
    for model_name in CONFIG['pso_models']:
        pred_path = os.path.join(pred_dir, f'{model_name}_test_preds.npy')
        if os.path.exists(pred_path):
            probs = np.load(pred_path)
            fpr, tpr, _ = roc_curve(y_test, probs)
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, color=colors.get(model_name, 'gray'),
                    label=f'{model_name.upper()} (AUC={roc_auc:.3f})', linewidth=1.2)

    # Ensemble curve
    ensemble_probs = np.zeros(len(y_test), dtype=np.float32)
    for model_name in CONFIG['pso_models']:
        pred_path = os.path.join(pred_dir, f'{model_name}_test_preds.npy')
        if os.path.exists(pred_path):
            ensemble_probs += weights.get(model_name, 0) * np.load(pred_path)

    fpr_g, tpr_g, _ = roc_curve(y_test, ensemble_probs)
    auc_g = auc(fpr_g, tpr_g)
    ax.plot(fpr_g, tpr_g, color='red', linewidth=2.5,
            label=f'Config G / Ensemble (AUC={auc_g:.3f})')
    ax.plot([0,1], [0,1], 'k--', alpha=0.4)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'AUC-ROC Curves â€” {dataset.capitalize()}')
    ax.legend(loc='lower right', fontsize=9)
    plt.tight_layout()
    out_path = os.path.join(project_root, CONFIG['figures_dir'], dataset, 'auc_roc_curve.png')
    fig.savefig(out_path, dpi=150)
    plt.close()
    print(f'Saved: {out_path}')

Saved: d:/WavSent-MTL/results/figures/kotekar\auc_roc_curve.png
Saved: d:/WavSent-MTL/results/figures/kaggle\auc_roc_curve.png


## Task 4.9 â€” SHAP Analysis on Best Config

In [8]:
import torch
from src.models.mtl_model import build_model
from src.evaluation.shap_analysis import run_shap_analysis

device = 'cpu'

for dataset in ['kotekar', 'kaggle']:
    proc_dir = os.path.join(project_root, CONFIG[f'{dataset}_processed_dir'])
    feat_path = os.path.join(proc_dir, 'selected_features.json')

    if not os.path.exists(feat_path):
        print(f'{dataset}: selected_features.json not found, skipping SHAP')
        continue

    with open(feat_path) as f:
        selected_features = json.load(f)

    X_train = np.load(os.path.join(proc_dir, 'X_train.npy'))
    X_val   = np.load(os.path.join(proc_dir, 'X_val.npy'))
    n_features = X_train.shape[2]

    # Build best model (BEST_REPR config = C or B, TKAN)
    # Use TKAN for SHAP (best config model)
    BEST_REPR = CONFIG['BEST_REPR']
    model_name = 'tkan'  # best individual model from ablation
    model = build_model(model_name, CONFIG, n_features)

    # Try to load saved model weights
    model_path = os.path.join(project_root, CONFIG['models_dir'], dataset, f'{model_name}_best.pt')
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location='cpu'))
        print(f'Loaded saved model: {model_path}')
    else:
        print(f'No saved model at {model_path} â€” using random weights for SHAP demo')

    model.eval()

    print(f'Running SHAP analysis: {dataset} | n_features={n_features}')
    shap_vals = run_shap_analysis(
        model=model,
        X_train=X_train,
        X_explain=X_val,
        feature_names=selected_features,
        dataset=dataset,
        n_background=min(100, len(X_train)),
        n_explain=min(200, len(X_val)),
        save_fig=True,
    )
    print(f'SHAP values shape: {shap_vals.shape}')

Loaded saved model: d:/WavSent-MTL/results/saved_models/kotekar\tkan_best.pt
Running SHAP analysis: kotekar | n_features=8
[SHAP DEBUG] type(shap_values)=<class 'numpy.ndarray'>
[SHAP DEBUG] array shape=(155, 5, 8, 1)
[SHAP DEBUG] shap_arr.shape after reshape=(155, 5, 8)
[SHAP DEBUG] mean_abs_shap.shape=(155, 8)
SHAP summary saved to d:/WavSent-MTL/results/figures/kotekar\shap_summary.png
SHAP values shape: (155, 8)
Loaded saved model: d:/WavSent-MTL/results/saved_models/kaggle\tkan_best.pt
Running SHAP analysis: kaggle | n_features=9
[SHAP DEBUG] type(shap_values)=<class 'numpy.ndarray'>
[SHAP DEBUG] array shape=(200, 5, 9, 1)
[SHAP DEBUG] shap_arr.shape after reshape=(200, 5, 9)
[SHAP DEBUG] mean_abs_shap.shape=(200, 9)
SHAP summary saved to d:/WavSent-MTL/results/figures/kaggle\shap_summary.png
SHAP values shape: (200, 9)


## Task 4.10 â€” Granger Causality Test

In [9]:
from statsmodels.tsa.stattools import grangercausalitytests

MAX_LAG = CONFIG['granger_max_lag']

for dataset in ['kotekar', 'kaggle']:
    proc_dir = os.path.join(project_root, CONFIG[f'{dataset}_processed_dir'])
    feat_csv = os.path.join(proc_dir, 'featured_data.csv')

    if not os.path.exists(feat_csv):
        print(f'{dataset}: featured_data.csv not found, skipping Granger')
        continue

    df_feat = pd.read_csv(feat_csv)
    df_feat['return'] = df_feat['Close'].pct_change()
    df_granger = df_feat[['return', 'polarity_mean']].dropna()

    print(f'\n--- {dataset.upper()} â€” Granger Causality: polarity_mean -> returns ---')
    gc_results = grangercausalitytests(
        df_granger[['return', 'polarity_mean']].values,
        maxlag=MAX_LAG, verbose=False
    )

    gc_rows = []
    for lag in range(1, MAX_LAG + 1):
        f_stat = gc_results[lag][0]['ssr_ftest'][0]
        p_val  = gc_results[lag][0]['ssr_ftest'][1]
        sig = 'Yes' if p_val < 0.05 else 'No'
        print(f'  Lag {lag}: F={f_stat:.4f}  p={p_val:.4f}  significant={sig}')
        gc_rows.append({'lag': lag, 'f_stat': f_stat, 'p_value': p_val, 'significant': sig})

    gc_df = pd.DataFrame(gc_rows)
    out_path = os.path.join(project_root, CONFIG['tables_dir'], dataset, 'granger_results.csv')
    gc_df.to_csv(out_path, index=False)
    print(f'Saved: {out_path}')


--- KOTEKAR â€” Granger Causality: polarity_mean -> returns ---
  Lag 1: F=0.6084  p=0.4356  significant=No
  Lag 2: F=0.3161  p=0.7290  significant=No
  Lag 3: F=0.3288  p=0.8045  significant=No
  Lag 4: F=0.2576  p=0.9051  significant=No
  Lag 5: F=0.2791  p=0.9247  significant=No
Saved: d:/WavSent-MTL/results/tables/kotekar\granger_results.csv

--- KAGGLE â€” Granger Causality: polarity_mean -> returns ---
  Lag 1: F=0.9698  p=0.3249  significant=No
  Lag 2: F=0.4552  p=0.6344  significant=No
  Lag 3: F=0.8587  p=0.4619  significant=No
  Lag 4: F=0.9600  p=0.4284  significant=No
  Lag 5: F=0.7839  p=0.5612  significant=No
Saved: d:/WavSent-MTL/results/tables/kaggle\granger_results.csv


## Tasks 4.11â€“4.12 â€” Trading Simulation + Sharpe

In [10]:
from src.evaluation.trading_sim import run_trading_simulation

trading_results = {}

for dataset in ['kotekar', 'kaggle']:
    pred_dir = os.path.join(ABLATION_DIR, dataset, 'val_predictions')
    weights_path = os.path.join(ABLATION_DIR, dataset, f'pso_weights_{dataset}.json')
    proc_dir = os.path.join(project_root, CONFIG[f'{dataset}_processed_dir'])
    feat_csv = os.path.join(proc_dir, 'featured_data.csv')

    if not os.path.exists(weights_path) or not os.path.exists(feat_csv):
        print(f'{dataset}: skipping trading sim (files not found)')
        continue

    with open(weights_path) as f:
        weights = json.load(f)

    df_feat = pd.read_csv(feat_csv)
    y_test  = np.load(os.path.join(proc_dir, 'y_clf_test.npy'))

    # Get test period close prices from temporal split
    from src.data.windows import temporal_split
    df_feat['Date'] = pd.to_datetime(df_feat['Date'])
    _, _, test_df = temporal_split(df_feat)
    close_test = test_df['Close'].values

    # Reconstruct ensemble test predictions
    ensemble_probs = np.zeros(len(y_test), dtype=np.float32)
    for model_name in CONFIG['pso_models']:
        pred_path = os.path.join(pred_dir, f'{model_name}_test_preds.npy')
        if os.path.exists(pred_path):
            ensemble_probs += weights.get(model_name, 0) * np.load(pred_path)

    print(f'\n--- {dataset.upper()} Trading Simulation (Config G) ---')
    sim_result = run_trading_simulation(
        y_prob=ensemble_probs,
        close_prices=close_test,
        dataset=dataset,
        config_name='G',
        save_fig=True,
    )
    trading_results[dataset] = sim_result

    # Save
    out_path = os.path.join(project_root, CONFIG['tables_dir'], dataset, 'trading_results.csv')
    pd.DataFrame([sim_result]).to_csv(out_path, index=False)
    print(f'Saved: {out_path}')

Split: Train=746 | Val=160 | Test=161

--- KOTEKAR Trading Simulation (Config G) ---
Trading simulation plot saved to d:/WavSent-MTL/results/figures/kotekar\trading_simulation.png
Sharpe=2.5248 | Cumulative=14.83% | N_trades=55 | Win rate=70.9%
Saved: d:/WavSent-MTL/results/tables/kotekar\trading_results.csv
Split: Train=1255 | Val=269 | Test=270

--- KAGGLE Trading Simulation (Config G) ---
Trading simulation plot saved to d:/WavSent-MTL/results/figures/kaggle\trading_simulation.png
Sharpe=2.0478 | Cumulative=24.74% | N_trades=147 | Win rate=65.3%
Saved: d:/WavSent-MTL/results/tables/kaggle\trading_results.csv


## Task 4.15 â€” Baselines (SVM + RF)

In [11]:
import subprocess
print('Running baselines...')
result = subprocess.run(
    ['python', os.path.join(project_root, 'baselines', 'run_baselines.py')],
    capture_output=True, text=True, cwd=project_root
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
# Also copy to results/tables/ for consistency
import shutil
for ds in ['kotekar', 'kaggle']:
    src = os.path.join(project_root, 'baselines', 'results', f'{ds}_baselines.csv')
    dst = os.path.join(project_root, CONFIG['tables_dir'], ds, 'baseline_results.csv')
    shutil.copy2(src, dst)
    print(f'Copied -> {dst}')


Running baselines...

Running baselines on kotekar
[kotekar] Training SVM...
  SVM test accuracy: 0.6026
[kotekar] Training Random Forest...
  RF test accuracy: 0.5321

Saved Ã¢â€ â€™ baselines/results/kotekar_baselines.csv
model dataset  accuracy  balanced_accuracy      auc  precision   recall       f1
  SVM kotekar  0.602564           0.570856 0.591410   0.610169 0.818182 0.699029
   RF kotekar  0.532051           0.555147 0.606534   0.647059 0.375000 0.474820

Running baselines on kaggle
[kaggle] Training SVM...
  SVM test accuracy: 0.5660
[kaggle] Training Random Forest...
  RF test accuracy: 0.6151

Saved Ã¢â€ â€™ baselines/results/kaggle_baselines.csv
model dataset  accuracy  balanced_accuracy      auc  precision   recall       f1
  SVM  kaggle  0.566038           0.575742 0.616586   0.674797 0.525316 0.590747
   RF  kaggle  0.615094           0.630457 0.677304   0.737288 0.550633 0.630435



## Final Summary vs Kotekar Benchmark

In [12]:
KOTEKAR_BENCHMARK_ACC   = 0.5853
KOTEKAR_BENCHMARK_SHARPE = 1.5679

print('=' * 65)
print('FINAL RESULTS vs BENCHMARK')
print('=' * 65)

for dataset in ['kotekar', 'kaggle']:
    df = results[dataset]
    cfg_G = df[df['config'] == 'G']
    if len(cfg_G) > 0:
        acc_G = cfg_G['accuracy'].values[0]
        auc_G = cfg_G['auc'].values[0]
    else:
        acc_G = auc_G = None

    sharpe_G = trading_results.get(dataset, {}).get('sharpe', None)

    print(f'\n{dataset.upper()}:')
    print(f'  Config G accuracy:  {acc_G:.4f}' if acc_G else '  Config G: not available')
    print(f'  Benchmark accuracy: {KOTEKAR_BENCHMARK_ACC}')
    if acc_G:
        delta = acc_G - KOTEKAR_BENCHMARK_ACC
        print(f'  Delta:              {delta:+.4f}  {"BEAT" if delta > 0 else "below"}')
    if sharpe_G:
        print(f'  Config G Sharpe:    {sharpe_G:.4f}')
        print(f'  Benchmark Sharpe:   {KOTEKAR_BENCHMARK_SHARPE}')

print()
print('All figures saved to:   results/figures/')
print('All tables saved to:    results/tables/')
print()
print('Phase 4 COMPLETE â€” ready for paper writing (Phase 5)')

FINAL RESULTS vs BENCHMARK

KOTEKAR:
  Config G accuracy:  0.5641
  Benchmark accuracy: 0.5853
  Delta:              -0.0212  below
  Config G Sharpe:    2.5248
  Benchmark Sharpe:   1.5679

KAGGLE:
  Config G accuracy:  0.6755
  Benchmark accuracy: 0.5853
  Delta:              +0.0902  BEAT
  Config G Sharpe:    2.0478
  Benchmark Sharpe:   1.5679

All figures saved to:   results/figures/
All tables saved to:    results/tables/

Phase 4 COMPLETE â€” ready for paper writing (Phase 5)
